In [82]:
import os
import shutil
import gdown

import pandas as pd
import numpy as np

In [83]:
# project_name = "predicting_electric_vehicle_purchases"
# project_dir = f"temp/{project_name}"

# os.makedirs(project_dir,exist_ok=True)
# url = "https://drive.google.com/file/d/1lZtYQt-uFIXmdoGSbQzN7TWNs2a-OdQp/view?usp=sharing"
# path = gdown.download(url=url, output=f"temp/{project_name}.zip")

# shutil.unpack_archive(path,project_dir)
# os.remove(path)

project_dir = "temp/predicting_electric_vehicle_purchases/"

In [164]:
train_data = pd.read_csv(os.path.join(project_dir,'train.csv')).drop(columns='id')
test_data = pd.read_csv(os.path.join(project_dir,"test.csv"))

target_feature = "Will_Buy_EV"

In [165]:
train_data.head(10)

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
5,50,95938.0,55.1,2,8,15,3.0,Other,Urban,Sedan,No,No,Low,No
6,55,109219.0,42.9,1,1,5,4.0,Male,Suburban,Sedan,Yes,Yes,Low,Yes
7,45,60796.0,52.2,2,13,3,2.0,Other,Urban,SUV,No,Yes,Medium,No
8,39,67725.0,31.5,2,11,3,2.0,Female,Urban,Sedan,No,Yes,Medium,No
9,57,114827.0,66.9,2,0,1,5.0,Male,Rural,Sedan,Yes,Yes,Low,Yes


In [166]:
train_data.shape, test_data.shape

((668665, 14), (286571, 14))

In [167]:
import mltoolhub as mlt

initial_summary = mlt.get_quick_summary(train_data, classify=True)
initial_summary

,feature,dtype,missing_count,missing_percentage,nature,skewness,skew_type,kurtosis,kurt_type,eveness,is_balanced,no_of_classes
0,Age,int64,0,0.0,numeric,-0.003786,normal,-1.174029,platy,NaN,False,NaN
1,Annual_Income_USD,float64,0,0.0,numeric,-0.013086,normal,-0.149234,platy,NaN,False,NaN
2,Daily_Commute_km,float64,0,0.0,numeric,-0.084589,normal,-1.060918,platy,NaN,False,NaN
3,Number_of_Cars_Owned,int64,0,0.0,category,NaN,NaN,NaN,NaN,0.747147,True,4.0
4,Charging_Stations_Near_Home,int64,0,0.0,numeric,0.698730,right-skewed,-0.482457,platy,NaN,False,NaN
5,Charging_Stations_Near_Work,int64,0,0.0,numeric,0.700188,right-skewed,-0.489245,platy,NaN,False,NaN
6,Environmental_Concern_Level,float64,0,0.0,category,NaN,NaN,NaN,NaN,0.999071,True,5.0
7,Gender,object,0,0.0,category,NaN,NaN,NaN,NaN,0.662519,True,3.0
8,City_Type,object,0,0.0,category,NaN,NaN,NaN,NaN,0.948974,True,3.0
9,Current_Car_Type,object,0,0.0,category,NaN,NaN,NaN,NaN,0.826562,True,4.0


In [168]:
train_data['Charging_Stations_Near_Work'].value_counts()

Charging_Stations_Near_Work
3     79739
1     53736
8     50675
9     49745
2     49538
6     49103
5     46737
7     45594
4     45472
0     25673
16    20049
17    18166
18    18113
14    17366
12    17308
13    17183
19    16934
15    16306
10    15616
11    15612
Name: count, dtype: int64

In [169]:
from sklearn.preprocessing import OrdinalEncoder


categorical_columns = initial_summary.loc[(initial_summary['dtype']=='object'), 'feature'].values.tolist()
col_orders = {"City_Type" : ['Rural','Suburban','Urban'],"Range_Anxiety_Level": ['Low','Medium','High']}

ordinal_columns = list(col_orders.keys())
ord_encoder = OrdinalEncoder(categories=list(col_orders.values()))

train_data[ordinal_columns] = ord_encoder.fit_transform(train_data[ordinal_columns])
test_data[ordinal_columns]= ord_encoder.transform(test_data[ordinal_columns])

In [170]:
nominal_columns = list(set(categorical_columns).difference(set(ordinal_columns).union(set([target_feature]))))

train_data = pd.get_dummies(train_data,columns=nominal_columns, drop_first=True, dtype = np.int64)
test_data = pd.get_dummies(test_data, columns=nominal_columns, drop_first=True, dtype = np.int64)

train_data[target_feature] = train_data[target_feature].map({"Yes":1, "No":0})

In [171]:
train_data

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,City_Type,Range_Anxiety_Level,Will_Buy_EV,Subsidy_Available_Yes,Home_Charging_Possible_Yes,Current_Car_Type_SUV,Current_Car_Type_Sedan,Current_Car_Type_Truck,Gender_Male,Gender_Other
0,66,92887.0,23.4,2,3,7,1.0,1.0,0.0,0,0,1,0,1,0,1,0
1,38,30000.0,5.0,1,2,2,4.0,0.0,0.0,0,0,1,1,0,0,1,0
2,26,94389.0,36.8,1,8,15,5.0,2.0,0.0,1,1,0,0,1,0,0,0
3,66,73580.0,23.7,2,6,9,3.0,1.0,0.0,0,0,1,0,0,0,1,0
4,54,57898.0,50.8,1,2,3,3.0,1.0,0.0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,30,71090.0,27.4,2,5,6,5.0,2.0,1.0,0,1,0,0,1,0,1,0
668661,32,129990.0,5.0,2,5,4,1.0,1.0,0.0,0,1,1,1,0,0,0,0
668662,64,121791.0,24.2,1,5,6,1.0,1.0,0.0,0,1,1,0,0,0,0,0
668663,51,115923.0,43.7,2,0,0,1.0,0.0,0.0,0,1,1,1,0,0,0,0


In [172]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = train_data.drop(columns = target_feature)
Y = train_data[target_feature]

X_train,X_test, Y_train, Y_test = train_test_split(X,Y,shuffle=True, random_state=43)

sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

X_train.shape, X_test.shape


((501498, 16), (167167, 16))

In [173]:

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from xgboost import XGBRegressor

model = XGBRegressor()
model.fit(X_train, Y_train)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [174]:
from sklearn.metrics import r2_score, root_mean_squared_error


y_pred = model.predict(X_test)


print(root_mean_squared_error(Y_test, y_pred), r2_score(Y_test, y_pred))

0.2665877342224121 0.5082472562789917


In [175]:
y_pred

array([-0.01143741,  0.43160725,  0.00699715, ...,  0.05830593,
       -0.00498993,  0.23903717], shape=(167167,), dtype=float32)

In [176]:
dict(zip(model.feature_names_in_,model.feature_importances_))

{np.str_('Age'): np.float32(0.004236074),
 np.str_('Annual_Income_USD'): np.float32(0.04170393),
 np.str_('Daily_Commute_km'): np.float32(0.005363729),
 np.str_('Number_of_Cars_Owned'): np.float32(0.0027032916),
 np.str_('Charging_Stations_Near_Home'): np.float32(0.0029157277),
 np.str_('Charging_Stations_Near_Work'): np.float32(0.0028910688),
 np.str_('Environmental_Concern_Level'): np.float32(0.31841832),
 np.str_('City_Type'): np.float32(0.0036988878),
 np.str_('Range_Anxiety_Level'): np.float32(0.15913838),
 np.str_('Subsidy_Available_Yes'): np.float32(0.43085355),
 np.str_('Home_Charging_Possible_Yes'): np.float32(0.014256313),
 np.str_('Current_Car_Type_SUV'): np.float32(0.002806402),
 np.str_('Current_Car_Type_Sedan'): np.float32(0.0026171906),
 np.str_('Current_Car_Type_Truck'): np.float32(0.002587215),
 np.str_('Gender_Male'): np.float32(0.002991507),
 np.str_('Gender_Other'): np.float32(0.0028184548)}

In [177]:
test_ids = test_data['id']
test_pred  = model.predict(test_data.drop(columns="id"))

In [178]:
test_pred

array([ 0.00826263,  0.03603784,  0.01725828, ...,  0.01311675,
       -0.00803789, -0.00469995], shape=(286571,), dtype=float32)

In [179]:
final_df = pd.DataFrame({"id": test_ids, "Will_Buy_EV": test_pred})

In [180]:
final_df

,id,Will_Buy_EV
0,668665,0.008263
1,668666,0.036038
2,668667,0.017258
3,668668,0.001007
4,668669,0.024751
...,...,...
286566,955231,0.012182
286567,955232,0.208379
286568,955233,0.013117
286569,955234,-0.008038


In [181]:
final_df.to_csv('XGBOOST_submission.csv', index=False)